# 04 — Error analysis: which photos does the model get wrong, and why?

Notebook 03 told us **how often** a model is wrong. This notebook shows **which photos** it got wrong
and **where on the face it was looking** when it made the mistake.

There are two kinds of mistake (fake = 1 is the thing we want to catch):

| Mistake | What happened | Why it matters in KYC |
|---|---|---|
| **False accept** | a **fake** face the model called real | fraud gets through (the `far` score) |
| **False reject** | a **real** customer the model called fake | a genuine person is blocked (the `frr` score) |

For the worst mistakes we draw a **Grad-CAM heatmap**: red areas are the parts of the face that pushed the model
towards "fake". If the red is on the background or the image edges instead of the face, the model may have learned
a shortcut instead of real signs of AI generation.

**Needs:** a trained model (`best.pt`, from notebook 05 or `train.py`) and the test images (notebook 01, Step 2).

**How to use:** run the cells from top to bottom (Shift+Enter). Works on your own computer, in Google Colab and on Kaggle.
A GPU makes Step 4 faster (on Colab: Runtime → Change runtime type → T4 GPU; on Kaggle: Settings → Accelerator).

### Running on Kaggle? One-time setup

In the notebook editor, open the right-hand panel:

- **Settings → Internet → On.** Needed to download the project code from GitHub. Kaggle asks you to verify your phone once.
- **Add Input → Datasets →** search **"140k Real and Fake Faces"** (by xhlulu) → **Add**. These are the images;
  Kaggle attaches them read-only under `/kaggle/input/`, so nothing is downloaded or unzipped.
- **Add Input → Your Work →** your saved notebook 05 (after **Save Version** there). This brings in its trained
  models (`outputs/<experiment>/best.pt`) and score files, read-only.
- **Settings → Accelerator → GPU T4 x2** (or P100). Makes the model steps much faster (free, about 30 hours a week).

The code finds everything by itself: `ON_KAGGLE` in Step 1 is `True` when running on Kaggle.
Files you create go to `/kaggle/working/`; they are kept only when you click **Save Version**.

## Step 1 — Find the project, the trained models and the tools

**What this does, in plain words** (same as notebooks 01–03):

1. Loads the Python tools we need.
2. Finds the project folder. On Colab and Kaggle it downloads (clones) it from GitHub first.
3. Installs the two extra packages this notebook needs if they are missing: `timm` (the model library)
   and `grad-cam` (the heatmaps). Colab already has PyTorch.
4. Makes the list of results folders: `<project>/outputs/`, plus on Colab your Google Drive folder
   `MyDrive/deepfake_kyc_outputs/` where notebook 05 saves, and on Kaggle the `outputs/` folders of notebooks you
   attached with **Add Input → Your Work**. Then lists the trained models (`best.pt`).

In [ ]:
# 1. Tools we need
import importlib.util
import random
import subprocess
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Where are we running? Each platform keeps files in a different place.
ON_COLAB = "google.colab" in sys.modules           # Google Colab
ON_KAGGLE = Path("/kaggle/input").exists()         # Kaggle notebook

# 2. Find the project folder: look in the current folder and every folder above it.
PROJECT_ROOT = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src" / "deepfake_kyc").is_dir():
        PROJECT_ROOT = folder
        break

# Not found (for example on Colab or Kaggle)? Download the project from GitHub into the current folder.
# (On Kaggle this needs Internet switched on: Settings -> Internet -> On.)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd() / "UOT_SCS_3546_041_Deep-Learning"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git",
                        str(PROJECT_ROOT)], check=True)
print("Project folder:", PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))   # lets Python import the project's own code

# 3. Install missing packages: {name used in "import": name used by pip}
needed = {"timm": "timm", "pytorch_grad_cam": "grad-cam"}
for module, package in needed.items():
    if importlib.util.find_spec(module) is None:
        print("Installing", package, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# 4. Folders to search for trained models.
RESULTS_FOLDERS = []
if ON_KAGGLE:
    # Outputs of your other Kaggle notebooks that you attached with Add Input (read-only),
    # e.g. notebook 05 after "Save Version". They look like /kaggle/input/<notebook>/.../outputs/
    for depth in range(1, 5):
        RESULTS_FOLDERS += sorted(Path("/kaggle/input").glob("/".join(["*"] * depth) + "/outputs"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")      # asks you to allow access to your Drive
    RESULTS_FOLDERS.append(Path("/content/drive/MyDrive/deepfake_kyc_outputs"))

RESULTS_FOLDERS.append(PROJECT_ROOT / "outputs")

checkpoints = []
for results_folder in RESULTS_FOLDERS:
    checkpoints += sorted(results_folder.glob("*/best.pt"))

print(f"Trained models found: {len(checkpoints)}")
for file in checkpoints:
    print("  ", file.parent.name, "  ", file)

## Step 2 — Make sure the test images are here

**What this does:** checks that the test photos exist in `data/raw/real_vs_fake/real-vs-fake/test/`.

**On Colab** every notebook runs on its **own** fresh computer, so the images you unpacked in notebook 01 are not
here. If they are missing, this cell copies `archive.zip` from your Google Drive (same file and folder as
notebook 01, Step 2a Drive) and unpacks **only the test photos**, which is much faster than the whole dataset.

**On Kaggle** the test photos come from the dataset you attached with Add Input, so nothing is copied.

**On your own computer** the images from notebook 01 are already in the project folder, so nothing is copied.

In [ ]:
DRIVE_ZIP = "/content/drive/MyDrive/archive.zip"   # same as notebook 01; change if your zip is elsewhere

from deepfake_kyc.config import load_config

cfg = load_config(PROJECT_ROOT / "configs" / "base.yaml")   # same settings file as notebook 01
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_ROOT = PROJECT_ROOT / cfg["data"]["root"]     # folder with train/ valid/ test/

# On Kaggle the dataset is attached read-only under /kaggle/input (Add Input -> "140k Real and Fake Faces"),
# so we use it from there instead of data/raw/. Nothing needs to be downloaded or unzipped.
def find_kaggle_dataset():
    """Look inside /kaggle/input for the folder called real-vs-fake that has train/ valid/ test/."""
    for depth in range(1, 5):                                  # look 1, 2, 3, then 4 folders deep
        pattern = "/".join(["*"] * depth) + "/real-vs-fake"    # e.g. "*/*/real-vs-fake"
        for folder in Path("/kaggle/input").glob(pattern):
            if (folder / "train").is_dir():
                return folder
    return None

if ON_KAGGLE and find_kaggle_dataset() is not None:
    DATA_ROOT = find_kaggle_dataset()
print("Images folder:", DATA_ROOT)
TEST_DIR = DATA_ROOT / "test"

def count_test_images():
    return len(list(TEST_DIR.glob("real/*.jpg"))), len(list(TEST_DIR.glob("fake/*.jpg")))

n_real, n_fake = count_test_images()
if (n_real == 0 or n_fake == 0) and ON_COLAB and not Path(DRIVE_ZIP).exists():
    print("Not found in your Drive:", DRIVE_ZIP, "-> put archive.zip there or change DRIVE_ZIP.")
elif (n_real == 0 or n_fake == 0) and ON_COLAB:
    print("Test images missing: unpacking only the test photos from", DRIVE_ZIP, "...")
    with zipfile.ZipFile(DRIVE_ZIP) as archive:
        test_files = [name for name in archive.namelist() if "/test/" in name]
        archive.extractall(RAW_DATA_DIR, members=test_files)
    n_real, n_fake = count_test_images()

print(f"Test images: real={n_real:,}  fake={n_fake:,}")
if n_real == 0 or n_fake == 0:
    print("Missing! On Kaggle: Add Input -> \"140k Real and Fake Faces\". Elsewhere: notebook 01, Step 2.")

## Step 3 — Pick one model and load it

**What this does:** loads one trained model. The settings it was trained with (model type, image size, …) are saved
inside `best.pt`, so the model is rebuilt exactly as it was trained. It also tells you how the model was trained:
on clean + randomly damaged photos (the project default) or on clean photos only (the `06_effnet_clean` control).

**Choose the model:** `CHECKPOINT = None` uses the newest one from Step 1. To pick one yourself, write its path, e.g.
`CHECKPOINT = PROJECT_ROOT / "outputs" / "03_effnet_finetune" / "best.pt"`.

In [ ]:
import torch

from deepfake_kyc.models import load_checkpoint
from deepfake_kyc.utils import get_device

CHECKPOINT = None      # None = newest model found in Step 1

if CHECKPOINT is None:
    CHECKPOINT = max(checkpoints, key=lambda path: path.stat().st_mtime)   # newest file

device = get_device()                                   # GPU if there is one, otherwise CPU
model, settings = load_checkpoint(CHECKPOINT, device)   # the model + the settings saved inside it
IMAGE_SIZE = settings["data"]["image_size"]

trained_with_damage = settings["augment"]["degradation"]
print("Model:     ", CHECKPOINT.parent.name, f"({settings['model']['name']})")
print("Trained on:", "clean + randomly damaged photos" if trained_with_damage else "clean photos only")
print("Image size:", IMAGE_SIZE, "x", IMAGE_SIZE)
print("Running on:", device)

## Step 4 — Run the model on the test photos

**What this does:** shows the model every test photo (real and fake mixed). For each photo the model gives a number
between 0 and 1, `P(fake)`: how sure it is that the face is AI-generated. If `P(fake)` is 0.5 or more we call it "fake".

**Controls:**
- `TEST_PHOTOS = 2000` uses 2,000 random test photos so it finishes quickly; `None` = all 20,000.
  With the same number, these are **the same photos** that notebook 03 (Step 1b) scored, so the numbers match.
- `THRESHOLD = 0.5`: the cut-off between "real" and "fake".

In [ ]:
from torch.utils.data import DataLoader

from deepfake_kyc.data import build_transform, make_dataset, subsample
from deepfake_kyc.engine import predict

TEST_PHOTOS = 2000     # None = all test photos
THRESHOLD = 0.5

# The same image preparation the model was tested with (resize, no random changes).
transform = build_transform(IMAGE_SIZE)
all_test_photos = make_dataset(TEST_DIR, transform=transform)
test_photos = subsample(all_test_photos, TEST_PHOTOS, settings["seed"])   # same pick as evaluate.py

# File path of every photo we use, in the same order as the predictions.
if test_photos is all_test_photos:
    paths = [path for path, _ in all_test_photos.samples]
else:
    paths = [all_test_photos.samples[i][0] for i in test_photos.indices]

loader = DataLoader(test_photos, batch_size=32, shuffle=False, num_workers=0)
probs, labels, _ = predict(model, loader, device)   # probs = P(fake) per photo, labels = true answer (1 = fake)
preds = (probs >= THRESHOLD).astype(int)          # the model's answer (1 = fake)

real = labels == 0
fake = labels == 1
print(f"Checked {len(labels):,} test photos.")
print(f"Real faces: {real.sum():,}  -> let through {(real & (preds == 0)).sum():,}, wrongly flagged {(real & (preds == 1)).sum():,}")
print(f"Fake faces: {fake.sum():,}  -> caught {(fake & (preds == 1)).sum():,}, missed {(fake & (preds == 0)).sum():,}")
print(f"Accuracy: {(preds == labels).mean():.3f}")

## Step 5 — Find the worst mistakes

**What this does:** collects the two kinds of mistake and sorts them so the **most confident** mistakes come first:

- worst **false accepts**: fake faces with the **lowest** `P(fake)` (the model was very sure they were real);
- worst **false rejects**: real faces with the **highest** `P(fake)` (the model was very sure they were fake).

In [ ]:
false_accepts = [i for i in range(len(labels)) if labels[i] == 1 and preds[i] == 0]   # fake called real
false_rejects = [i for i in range(len(labels)) if labels[i] == 0 and preds[i] == 1]   # real called fake

false_accepts.sort(key=lambda i: probs[i])      # lowest P(fake) first
false_rejects.sort(key=lambda i: -probs[i])     # highest P(fake) first

print(f"False accepts (fake -> real): {len(false_accepts)}")
print(f"False rejects (real -> fake): {len(false_rejects)}")

## Step 6 — Look at the mistakes with Grad-CAM heatmaps

**What this does:** for each of the worst mistakes, shows the photo (left) and the same photo with the heatmap on
top (right). **Red/yellow** = the area that pushed the model towards "fake"; **blue** = little influence.

**What to look for:**
- Is the red on the **face** (eyes, mouth, skin, hair edges)? That is where AI artefacts usually are.
- Is the red on the **background** or the **image border**? Then the model may be using a shortcut.
- Do the mistakes have something in common (glasses, smiles, dark photos, children, …)?

**Caution:** a heatmap is a rough hint, not proof. It is low-resolution and can look convincing even when it is wrong.

Change `NUM_TO_SHOW` to see more or fewer of each kind.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget

from deepfake_kyc.models import gradcam_target_layer

NUM_TO_SHOW = 4        # how many of each kind of mistake

# Grad-CAM looks at the model's last convolution layer to see which areas mattered most.
cam = GradCAM(model=model, target_layers=[gradcam_target_layer(model)])

def heatmap(image):
    """Return (P(fake), photo with the heatmap drawn on top)."""
    image = image.convert("RGB")
    x = transform(image).unsqueeze(0).to(device)               # prepare the photo like in Step 4
    with torch.no_grad():
        p_fake = torch.sigmoid(model(x)).item()
    heat = cam(input_tensor=x, targets=[BinaryClassifierOutputTarget(1)])[0]   # 1 = "fake"
    photo = np.asarray(image.resize((IMAGE_SIZE, IMAGE_SIZE)), dtype=np.float32) / 255
    return p_fake, show_cam_on_image(photo, heat, use_rgb=True)

to_show = [(i, "false accept: fake called real") for i in false_accepts[:NUM_TO_SHOW]] + \
          [(i, "false reject: real called fake") for i in false_rejects[:NUM_TO_SHOW]]

if to_show:
    fig, axes = plt.subplots(len(to_show), 2, figsize=(6, 3 * len(to_show)), squeeze=False)
    for row, (i, kind) in zip(axes, to_show):
        image = Image.open(paths[i])
        p_fake, overlay = heatmap(image)
        row[0].imshow(image.convert("RGB"))
        row[0].set_title(kind, fontsize=9)
        row[1].imshow(overlay)
        row[1].set_title(f"P(fake) = {p_fake:.2f}", fontsize=9)
        row[0].axis("off")
        row[1].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No mistakes to show: the model got every checked photo right.")

## Step 7 (try it) — Does damage change where the model looks?

**What this does:** takes one test photo and shows the heatmap on the clean photo and on damaged versions of it,
using the same damage code as notebook 02. If the red area jumps around or the `P(fake)` changes a lot,
that damage confuses the model. Models trained on clean + damaged photos should change less than the
`06_effnet_clean` control.

Change `PHOTO`, `DAMAGE` and run the cell again.

In [ ]:
from deepfake_kyc.degradations import DEGRADATIONS

PHOTO = 0              # which photo: a number from 0 to the number of checked photos - 1
DAMAGE = "blur"        # try "jpeg", "blur", "downscale", "noise" or "screen"

image = Image.open(paths[PHOTO]).convert("RGB")
truth = "fake" if labels[PHOTO] == 1 else "real"
versions = [("clean", image)] + [(f"{DAMAGE} {s}", DEGRADATIONS[DAMAGE](s)(image)) for s in [1, 3, 5]]

fig, axes = plt.subplots(2, len(versions), figsize=(3 * len(versions), 6))
for col, (name, version) in enumerate(versions):
    p_fake, overlay = heatmap(version)
    axes[0, col].imshow(version)
    axes[0, col].set_title(name)
    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f"P(fake) = {p_fake:.2f}")
    axes[0, col].axis("off")
    axes[1, col].axis("off")
fig.suptitle(f"One {truth} test photo: clean vs. {DAMAGE} at severity 1, 3, 5")
plt.tight_layout()
plt.show()

## Next

- Compare these mistakes with another model: change `CHECKPOINT` in Step 3 (for example E3 vs. the
  `06_effnet_clean` control) and run Steps 3–7 again.
- The overall scores for every model are in [03_results_charts.ipynb](03_results_charts.ipynb).

## Quick notes to remember

- This notebook only **reads** files; it saves nothing.
- A false accept costs fraud; a false reject costs a customer. In the report, show a few of each.
- Grad-CAM is a rough guide, not proof. Say this in the report.
- On Colab, each notebook has its own fresh computer: Step 2 copies the test photos again each new session.